In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
import time

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data paths and transformations
data_path = "Dataset/Tulsi/train_aug"

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# Load dataset
dataset = datasets.ImageFolder(root=data_path, transform=transform)


# Train-test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# CNN Model
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Model, Loss, and Optimizer
num_classes = 8  # Updated to match the number of classes in the tulsi dataset
model = CNNModel(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Evaluation function
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            val_loss += loss.item()
    accuracy = 100 * correct / total
    avg_val_loss = val_loss / len(test_loader)
    return accuracy, avg_val_loss

# Training loop
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        start_time = time.time()
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            if (i + 1) % 50 == 0:
                print(f'{i+1}/{len(train_loader)} ━ Training...')
        
        accuracy, val_loss = evaluate_model(model, test_loader)
        elapsed_time = time.time() - start_time
        print(f'Epoch {epoch+1}/{num_epochs} ━ {len(train_loader)}/{len(train_loader)} ━ {elapsed_time:.2f}s - accuracy: {accuracy:.4f} - loss: {total_loss / len(train_loader):.4f} - val_accuracy: {accuracy:.4f} - val_loss: {val_loss:.4f}')

# Train and evaluate
train_model(model, train_loader, criterion, optimizer, num_epochs=10)

# Save model
torch.save(model.state_dict(), 'tul_leaf_disease_cnn.pth')  # Model name updated
print('Model saved successfully.')


50/57 ━ Training...
Epoch 1/10 ━ 57/57 ━ 53.88s - accuracy: 72.0879 - loss: 0.9660 - val_accuracy: 72.0879 - val_loss: 0.5879
50/57 ━ Training...
Epoch 2/10 ━ 57/57 ━ 25.39s - accuracy: 91.8681 - loss: 0.4500 - val_accuracy: 91.8681 - val_loss: 0.2307
50/57 ━ Training...
Epoch 3/10 ━ 57/57 ━ 30.44s - accuracy: 95.6044 - loss: 0.1333 - val_accuracy: 95.6044 - val_loss: 0.1298
50/57 ━ Training...
Epoch 4/10 ━ 57/57 ━ 29.68s - accuracy: 97.1429 - loss: 0.0614 - val_accuracy: 97.1429 - val_loss: 0.0933
50/57 ━ Training...
Epoch 5/10 ━ 57/57 ━ 29.86s - accuracy: 91.6484 - loss: 0.0275 - val_accuracy: 91.6484 - val_loss: 0.2836
50/57 ━ Training...
Epoch 6/10 ━ 57/57 ━ 32.15s - accuracy: 96.4835 - loss: 0.0271 - val_accuracy: 96.4835 - val_loss: 0.0886
50/57 ━ Training...
Epoch 7/10 ━ 57/57 ━ 38.30s - accuracy: 98.6813 - loss: 0.0138 - val_accuracy: 98.6813 - val_loss: 0.0342
50/57 ━ Training...
Epoch 8/10 ━ 57/57 ━ 36.17s - accuracy: 99.5604 - loss: 0.0089 - val_accuracy: 99.5604 - val_loss:

: 